# Customer Shopping Behaviour — Data Cleaning

**Project:** End-to-End Retail Customer Behaviour Analytics  
**Author:** Daniel Sampson  
**Tool:** Python (Pandas)

---

## Objective
Clean the raw dataset by handling missing values, standardising
column names, checking for redundancies, and exporting the
cleaned data ready for feature engineering and SQL analysis.

In [1]:
import pandas as pd

# Load raw dataset
df = pd.read_csv('../data/customer_shopping_behavior.csv')

print(f'Shape before cleaning: {df.shape}')

Shape before cleaning: (3900, 18)


## 1. Column Name Standardisation

Rename all columns to snake_case for consistency across
Python, SQL Server, and Power BI.

In [2]:
# Convert to lowercase and replace spaces with underscores
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Rename the purchase amount column to remove the (usd) suffix
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

print('Cleaned column names:')
print(df.columns.tolist())

Cleaned column names:
['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases']


## 2. Missing Value Handling

37 null values found in `review_rating`.
Strategy: impute using the **median rating per product category**
to preserve category-specific rating behaviour.

In [3]:
# Impute missing review_rating with median per category
df['review_rating'] = df.groupby('category')['review_rating'].transform(
    lambda x: x.fillna(x.median())
)

# Confirm no more nulls
print('Missing values after imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nNo missing values!' if df.isnull().sum().sum() == 0 else 'Still has nulls.')

Missing values after imputation:
Series([], dtype: int64)

No missing values!


## 3. Duplicate Check

In [4]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

# Drop duplicates if any exist
if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')

Duplicate rows: 0


## 4. Data Consistency Check — Redundant Columns

Verify if `discount_applied` and `promo_code_used` carry
identical information. If so, one column can be safely dropped.

In [5]:
# Check if the two columns are identical row by row
are_identical = (df['discount_applied'] == df['promo_code_used']).all()
print(f'discount_applied == promo_code_used for all rows: {are_identical}')

# Since they are 100% identical, drop the redundant column
df = df.drop(columns=['promo_code_used'])
print('promo_code_used dropped. Remaining columns:', df.shape[1])

discount_applied == promo_code_used for all rows: True
promo_code_used dropped. Remaining columns: 17


## 5. Data Types Verification

In [6]:
# Confirm final data types are appropriate
df.dtypes

customer_id                 int64
age                         int64
gender                     object
item_purchased             object
category                   object
purchase_amount             int64
location                   object
size                       object
color                      object
season                     object
review_rating             float64
subscription_status        object
shipping_type              object
discount_applied           object
previous_purchases          int64
payment_method             object
frequency_of_purchases     object
dtype: object

In [7]:
# Final shape after all cleaning steps
print(f'Shape after cleaning: {df.shape}')
df.head()

Shape after cleaning: (3900, 17)


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually


## 6. Export Cleaned Dataset

In [8]:
# Save cleaned dataset to the data folder
df.to_csv('../data/customer_data_cleaned.csv', index=False)
print('Cleaned dataset saved to: ../data/customer_data_cleaned.csv')

Cleaned dataset saved to: ../data/customer_data_cleaned.csv


## 7. Summary of Cleaning Steps

| Step | Action | Result |
|---|---|---|
| Column names | Converted to snake_case | Consistent naming across tools |
| Missing values | Median imputation per category | 37 nulls in review_rating resolved |
| Duplicates | Checked | None found |
| Redundant column | promo_code_used dropped | 100% identical to discount_applied |
| Export | Saved as CSV | Ready for feature engineering & SQL |

---
*Next: `03_feature_engineering.ipynb`*